# Online Inference with FastAPI

The batch flow handles scheduled scoring. This notebook covers the second serving pattern in the repo: online inference through a small FastAPI application.

Before you run it, finish [02-train-ml-model.ipynb](02-train-ml-model.ipynb) so the API can resolve a registered MLflow model version.


## Workflow

```mermaid
flowchart LR
    A["Client<br>request"] --> B["FastAPI /predict<br>endpoint"]
    B --> C["Resolve model<br>from MLflow"]
    C --> D["Predict one request<br>at a time"]
    D --> E["Return JSON<br>response"]
```


In [1]:
import pandas as pd
from fastapi.testclient import TestClient

from src.serve.api import app

client = TestClient(app)
client.get("/health").json()

{'status': 'ok'}

## Send a Sample Prediction Request

The API expects the same logical inputs used in training and batch scoring: pickup location, dropoff location, and trip distance.

In [2]:
sample_payload = {
    "PULocationID": 1,
    "DOLocationID": 2,
    "trip_distance": 3.5,
}

response = client.post("/predict", json=sample_payload)
response.json()

{'prediction': 15.437701274535783,
 'model_uri': 'models:/green-taxi-duration/2'}

## Compare a Few Scenarios

This quick comparison is a useful way to check whether the API output moves in the direction you would expect when distance changes.

The API loads the active MLflow model once and then reuses it across requests, which matches a common production pattern for lightweight inference services.

In [3]:
scenarios = [
    {
        "label": "short city trip",
        "PULocationID": 1,
        "DOLocationID": 2,
        "trip_distance": 1.2,
    },
    {
        "label": "medium cross-town trip",
        "PULocationID": 1,
        "DOLocationID": 2,
        "trip_distance": 3.5,
    },
    {
        "label": "longer trip",
        "PULocationID": 1,
        "DOLocationID": 2,
        "trip_distance": 7.0,
    },
]

scenario_results = []
for scenario in scenarios:
    payload = {k: v for k, v in scenario.items() if k != "label"}
    prediction = client.post("/predict", json=payload).json()["prediction"]
    scenario_results.append({"scenario": scenario["label"], "prediction": prediction})

pd.DataFrame(scenario_results)

,scenario,prediction
0,short city trip,9.489691
1,medium cross-town trip,15.437701
2,longer trip,24.489021


## Run the Live Service

The notebook uses `TestClient` so you can experiment without starting a separate server process. If you want the live API, run:

```bash
uv run python -m uvicorn src.serve.api:app --reload
```

Then try a real HTTP request:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"PULocationID": 1, "DOLocationID": 2, "trip_distance": 3.5}'
```

If the request returns a model-resolution error, go back to the training notebook and confirm that the MLflow server is still reachable at the URL from `.env`.
